In [1]:

!pip install protobuf==3.20.3 --force-reinstall --no-cache-dir


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 6.1 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 6.33.0
    Uninstalling protobuf-6.33.0:
      Successfully uninstalled protobuf-6.33.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
onnx 1.18.0 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
a2a-sdk 0.3.10 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
ray 2.51.1 requires click!=8.3.0,>=7.0, but you have click 8.3.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
tensorflow-me

In [2]:
# Monet CycleGAN (ResNet + Augmentation)

import os, random, zipfile
from glob import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras import mixed_precision

print("TensorFlow:", tf.__version__)

tpu_available = False
try:
    resolver = tf.distribute.cluster_resolver.TPUClusterResolver.connect()
    strategy = tf.distribute.TPUStrategy(resolver)
    tpu_available = True
except ValueError:
    strategy = tf.distribute.get_strategy()

# configure
IMG_HEIGHT = 256
IMG_WIDTH = 256
IMG_CHANNELS = 3
BUFFER_SIZE = 1024
BATCH_SIZE = 4
EPOCHS = 25
LAMBDA_CYCLE = 10.0
LAMBDA_ID = 0.5
NUM_GENERATED_IMAGES = 7000

MONET_TFREC_PATTERN = "/kaggle/input/gan-getting-started/monet_tfrec/*.tfrec"
PHOTO_TFREC_PATTERN = "/kaggle/input/gan-getting-started/photo_tfrec/*.tfrec"

OUTPUT_DIR = "/kaggle/working/monet_generated"
os.makedirs(OUTPUT_DIR, exist_ok=True)

SEED = 1337
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# data
IMAGE_FEATURE_DESCRIPTION = {
    "image_name": tf.io.FixedLenFeature([], tf.string),
    "image": tf.io.FixedLenFeature([], tf.string),
}

def parse_tfrecord(example):
    ex = tf.io.parse_single_example(example, IMAGE_FEATURE_DESCRIPTION)
    img = tf.image.decode_jpeg(ex["image"], channels=IMG_CHANNELS)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    img = (tf.cast(img, tf.float32) / 127.5) - 1
    return img

def random_jitter(img):
    img = tf.image.resize(img, [286, 286])
    img = tf.image.random_crop(img, [IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
    img = tf.image.random_flip_left_right(img)
    return img

def load_dataset(pattern, batch, augment=False):
    files = tf.io.gfile.glob(pattern)
    ds = tf.data.TFRecordDataset(files, num_parallel_reads=tf.data.AUTOTUNE)
    ds = ds.map(parse_tfrecord, num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(lambda x: random_jitter(x), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.shuffle(BUFFER_SIZE).repeat().batch(batch).prefetch(tf.data.AUTOTUNE)
    return ds

# model comp
class InstanceNormalization(tf.keras.layers.Layer):
    def __init__(self, eps=1e-5, **kw):
        super().__init__(**kw); self.eps=eps
    def build(self, shape):
        c = shape[-1]
        self.gamma = self.add_weight(shape=(c,), initializer="ones", trainable=True)
        self.beta = self.add_weight(shape=(c,), initializer="zeros", trainable=True)
    def call(self, x):
        m,v = tf.nn.moments(x, [1,2], keepdims=True)
        x = (x - m) / tf.sqrt(v + self.eps)
        return self.gamma*x + self.beta

def resnet_block(x, f):
    init = tf.random_normal_initializer(0., 0.02)
    y = tf.keras.layers.Conv2D(f, 3, padding='same', kernel_initializer=init, use_bias=False)(x)
    y = InstanceNormalization()(y); y = tf.keras.layers.ReLU()(y)
    y = tf.keras.layers.Conv2D(f, 3, padding='same', kernel_initializer=init, use_bias=False)(y)
    y = InstanceNormalization()(y)
    return tf.keras.layers.Add()([x, y])

def build_resnet_generator(num_blocks=9):
    init = tf.random_normal_initializer(0., 0.02)
    inp = tf.keras.layers.Input([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])

    x = tf.keras.layers.Conv2D(64, 7, padding='same', kernel_initializer=init, use_bias=False)(inp)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(128, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(256, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    for _ in range(num_blocks):
        x = resnet_block(x, 256)

    x = tf.keras.layers.Conv2DTranspose(128, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2DTranspose(64, 3, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.ReLU()(x)

    x = tf.keras.layers.Conv2D(3, 7, padding='same', kernel_initializer=init)(x)
    out = tf.keras.layers.Activation('tanh')(x)
    return tf.keras.Model(inp, out, name="ResNetGen")

def build_discriminator():
    init = tf.random_normal_initializer(0., 0.02)
    inp = tf.keras.layers.Input([IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS])
    x = tf.keras.layers.Conv2D(64, 4, strides=2, padding='same', kernel_initializer=init)(inp)
    x = tf.keras.layers.LeakyReLU(0.2)(x)

    for f in [128, 256]:
        x = tf.keras.layers.Conv2D(f, 4, strides=2, padding='same', kernel_initializer=init, use_bias=False)(x)
        x = InstanceNormalization()(x); x = tf.keras.layers.LeakyReLU(0.2)(x)

    x = tf.keras.layers.Conv2D(512, 4, padding='same', kernel_initializer=init, use_bias=False)(x)
    x = InstanceNormalization()(x); x = tf.keras.layers.LeakyReLU(0.2)(x)

    out = tf.keras.layers.Conv2D(1, 4, padding='same', kernel_initializer=init)(x)
    return tf.keras.Model(inp, out, name="PatchDisc")

with strategy.scope():
    if tpu_available:
        mixed_precision.set_global_policy("mixed_bfloat16")

    # loss calc
    gan_loss = tf.keras.losses.MeanSquaredError()

    def generator_gan_loss(fake):
        return gan_loss(tf.ones_like(fake), fake)

    def discriminator_loss(real, fake):
        return 0.5*(gan_loss(tf.ones_like(real), real) + gan_loss(tf.zeros_like(fake), fake))

    def cycle_loss(r, c):
        r = tf.cast(r, tf.float32)
        c = tf.cast(c, tf.float32)
        return LAMBDA_CYCLE * tf.reduce_mean(tf.abs(r - c))

    def identity_loss(r, s):
        r = tf.cast(r, tf.float32)
        s = tf.cast(s, tf.float32)
        return LAMBDA_CYCLE * LAMBDA_ID * tf.reduce_mean(tf.abs(r - s))


    generator_g = build_resnet_generator()
    generator_f = build_resnet_generator()
    discriminator_x = build_discriminator()
    discriminator_y = build_discriminator()

    lr = 2e-4
    g_g_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    g_f_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    d_x_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)
    d_y_opt = tf.keras.optimizers.Adam(lr, beta_1=0.5)

# data
monet_ds = load_dataset(MONET_TFREC_PATTERN, BATCH_SIZE, augment=True)
photo_ds = load_dataset(PHOTO_TFREC_PATTERN, BATCH_SIZE, augment=True)
train_ds = tf.data.Dataset.zip((photo_ds, monet_ds))
train_ds = strategy.experimental_distribute_dataset(train_ds)

def train_step(real_x, real_y):
    with tf.GradientTape(persistent=True) as tape:
        fake_y = generator_g(real_x, training=True)
        fake_x = generator_f(real_y, training=True)

        cycled_x = generator_f(fake_y, training=True)
        cycled_y = generator_g(fake_x, training=True)

        same_x = generator_f(real_x, training=True)
        same_y = generator_g(real_y, training=True)

        dx_real = discriminator_x(real_x, training=True)
        dy_real = discriminator_y(real_y, training=True)

        dx_fake = discriminator_x(fake_x, training=True)
        dy_fake = discriminator_y(fake_y, training=True)

        g_adv = generator_gan_loss(dy_fake)
        f_adv = generator_gan_loss(dx_fake)

        cyc = cycle_loss(real_x, cycled_x) + cycle_loss(real_y, cycled_y)
        idg = identity_loss(real_y, same_y)
        idf = identity_loss(real_x, same_x)

        g_total = g_adv + cyc + idg
        f_total = f_adv + cyc + idf

        dx_loss = discriminator_loss(dx_real, dx_fake)
        dy_loss = discriminator_loss(dy_real, dy_fake)

    g_g_opt.apply_gradients(zip(tape.gradient(g_total, generator_g.trainable_variables), generator_g.trainable_variables))
    g_f_opt.apply_gradients(zip(tape.gradient(f_total, generator_f.trainable_variables), generator_f.trainable_variables))
    d_x_opt.apply_gradients(zip(tape.gradient(dx_loss, discriminator_x.trainable_variables), discriminator_x.trainable_variables))
    d_y_opt.apply_gradients(zip(tape.gradient(dy_loss, discriminator_y.trainable_variables), discriminator_y.trainable_variables))

    return g_total, f_total, dx_loss, dy_loss

@tf.function
def distributed_train_step(dist_inputs):
    def step_fn(inputs):
        real_x, real_y = inputs
        return train_step(real_x, real_y)

    g_total, f_total, dx_loss, dy_loss = strategy.run(step_fn, args=(dist_inputs,))
    g_total = strategy.reduce(tf.distribute.ReduceOp.MEAN, g_total, axis=None)
    f_total = strategy.reduce(tf.distribute.ReduceOp.MEAN, f_total, axis=None)
    dx_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, dx_loss, axis=None)
    dy_loss = strategy.reduce(tf.distribute.ReduceOp.MEAN, dy_loss, axis=None)
    return g_total, f_total, dx_loss, dy_loss

# training
for batch in train_ds.take(1):
    print(distributed_train_step(batch))
    break

STEPS_PER_EPOCH = 500

for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    for step, batch in enumerate(train_ds.take(STEPS_PER_EPOCH), 1):
        g1, g2, d1, d2 = distributed_train_step(batch)
        if step % 100 == 0:
            print(f"Step {step}: Gg={g1:.3f} Gf={g2:.3f} Dx={d1:.3f} Dy={d2:.3f}")

print("\nTraining done!")

# generation
def tensor_to_img(t):
    t = (t + 1)*127.5
    t = tf.clip_by_value(t,0,255)
    return tf.cast(t, tf.uint8).numpy()

def save_img(t, path):
    img = tf.keras.preprocessing.image.array_to_img(tensor_to_img(t))
    img.save(path, "JPEG")

print("\nGenerating Monet images...")
gen_ds = load_dataset(PHOTO_TFREC_PATTERN, 1, augment=False)

n = 0
for batch in gen_ds:
    fake = generator_g(batch, training=False)
    for img in fake:
        if n >= NUM_GENERATED_IMAGES:
            break
        save_img(img, f"{OUTPUT_DIR}/image_{n:05d}.jpg")
        n += 1
    if n >= NUM_GENERATED_IMAGES:
        break

print("Saved:", n)

# zip
zip_path = "/kaggle/working/images.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in os.listdir(OUTPUT_DIR):
        if f.endswith(".jpg"):
            z.write(os.path.join(OUTPUT_DIR, f), f)

print("Zipped:", zip_path)


2025-12-29 04:11:48.582970: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1766981508.784172      20 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1766981508.839442      20 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


TensorFlow: 2.18.0


I0000 00:00:1766981526.291128      20 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
I0000 00:00:1766981596.212941      65 cuda_dnn.cc:529] Loaded cuDNN version 90300


(<tf.Tensor: shape=(), dtype=float32, numpy=17.389720916748047>, <tf.Tensor: shape=(), dtype=float32, numpy=18.203947067260742>, <tf.Tensor: shape=(), dtype=float32, numpy=1.5864338874816895>, <tf.Tensor: shape=(), dtype=float32, numpy=1.621030569076538>)

Epoch 1/25
Step 100: Gg=7.061 Gf=7.296 Dx=0.252 Dy=0.289
Step 200: Gg=7.254 Gf=7.915 Dx=0.176 Dy=0.237
Step 300: Gg=5.280 Gf=5.367 Dx=0.266 Dy=0.316
Step 400: Gg=5.070 Gf=5.319 Dx=0.233 Dy=0.216
Step 500: Gg=5.784 Gf=5.964 Dx=0.180 Dy=0.310

Epoch 2/25
Step 100: Gg=5.408 Gf=4.912 Dx=0.301 Dy=0.261
Step 200: Gg=5.185 Gf=4.829 Dx=0.198 Dy=0.190
Step 300: Gg=4.432 Gf=4.015 Dx=0.232 Dy=0.539
Step 400: Gg=4.915 Gf=5.449 Dx=0.204 Dy=0.241
Step 500: Gg=4.440 Gf=4.374 Dx=0.174 Dy=0.154

Epoch 3/25
Step 100: Gg=4.435 Gf=4.147 Dx=0.182 Dy=0.252
Step 200: Gg=4.989 Gf=4.628 Dx=0.140 Dy=0.178
Step 300: Gg=4.075 Gf=4.240 Dx=0.215 Dy=0.221
Step 400: Gg=4.340 Gf=4.633 Dx=0.251 Dy=0.158
Step 500: Gg=4.402 Gf=4.090 Dx=0.263 Dy=0.250

Epoch 4/25
Step 1